# PaDiM — Baseline anomaly detection sur `cable` (MVTec AD)

Implémentation **from-scratch** de PaDiM (Patch Distribution Modeling) sur la catégorie `cable` du dataset MVTec AD. Cette baseline servira de point de référence pour décider du préprocessing à mettre en place.

**Sommaire :**
1. Setup & dataset
2. Feature extractor (ResNet18 + hooks)
3. PaDiM — fit & score
4. Entraînement sur `cable` (train good)
5. Inférence sur le test set
6. Évaluation (AUROC image + pixel)
7. Visualisation des heatmaps
8. Analyse → décisions de préprocessing

**Pré-requis :** `uv run python -m src.data.harmonize` doit avoir été lancé.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA, EDA, PATHS

warnings.filterwarnings('ignore')
sns.set_theme(style=EDA.sns_style, palette=EDA.sns_palette, font_scale=EDA.sns_font_scale)
plt.rcParams['figure.dpi'] = EDA.figure_dpi

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = DATA.random_seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# --- PaDiM hyperparams ---
CATEGORY = 'cable'
IMG_SIZE = 256
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
BACKBONE = 'resnet18'
D_PROJ = 100
EPSILON = 0.01
SMOOTH_SIGMA = 4.0

print(f'Device : {DEVICE}')
print(f'Catégorie : {CATEGORY} | image : {IMG_SIZE}x{IMG_SIZE} | backbone : {BACKBONE}')

## 1. Dataset & transforms

On charge le CSV unifié, on filtre sur `cable` (MVTec). On définit un `Dataset` minimal qui renvoie `(image, mask, label)` après resize + normalisation ImageNet.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

df = pd.read_csv(PATHS.unified_csv)
df_cat = df[(df['dataset'] == DATA.mvtec_name) & (df['category'] == CATEGORY)].reset_index(drop=True)
print(f'{len(df_cat)} images pour {CATEGORY}')
print(df_cat.groupby(['split', 'is_anomaly']).size())

img_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
mask_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

class AnomalyDataset(Dataset):
    def __init__(self, df, img_tfm, mask_tfm, root):
        self.df = df.reset_index(drop=True)
        self.img_tfm = img_tfm
        self.mask_tfm = mask_tfm
        self.root = Path(root)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['image_path']).convert('RGB')
        x = self.img_tfm(img)
        if isinstance(row.get('mask_path'), str) and bool(row['has_mask']):
            m = Image.open(self.root / row['mask_path']).convert('L')
            m = (self.mask_tfm(m) > 0).float()
        else:
            m = torch.zeros(1, IMG_SIZE, IMG_SIZE)
        return {
            'image': x,
            'mask': m,
            'label': int(row['is_anomaly']),
            'image_path': row['image_path'],
            'defect_label': row.get('label', 'good'),
        }

train_df = df_cat[(df_cat['split'] == 'train') & (~df_cat['is_anomaly'])]
test_df  = df_cat[df_cat['split'] == 'test']

train_ds = AnomalyDataset(train_df, img_tfm, mask_tfm, PATHS.root)
test_ds  = AnomalyDataset(test_df,  img_tfm, mask_tfm, PATHS.root)

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

print(f'\nTrain (good) : {len(train_ds)} | Test : {len(test_ds)}')

In [ ]:
# Sanity check : afficher 3 train + 3 test (1 good + 2 anomal) après preprocessing
def denorm(x):
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std  = np.array(IMAGENET_STD).reshape(3, 1, 1)
    return np.clip(x.cpu().numpy() * std + mean, 0, 1).transpose(1, 2, 0)

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for c, i in enumerate(range(3)):
    s = train_ds[i]
    axes[0, c].imshow(denorm(s['image']))
    axes[0, c].set_title(f"TRAIN • {s['defect_label']}", fontsize=10)
    axes[0, c].axis('off')

anom_test_idx = test_df[test_df['is_anomaly']].index.tolist()
good_test_idx = test_df[~test_df['is_anomaly']].index.tolist()
picks = [good_test_idx[0]] + anom_test_idx[:2]
for c, idx in enumerate(picks):
    pos = test_df.index.get_loc(idx)
    s = test_ds[pos]
    axes[1, c].imshow(denorm(s['image']))
    if s['mask'].sum() > 0:
        axes[1, c].imshow(s['mask'][0].numpy(), cmap='Reds', alpha=0.4)
    axes[1, c].set_title(f"TEST • {s['defect_label']}", fontsize=10)
    axes[1, c].axis('off')

plt.tight_layout()
plt.show()

## 2. Feature extractor (ResNet18 + hooks)

Backbone gelé. On enregistre des hooks sur `layer1`, `layer2`, `layer3` ; on upsample `layer2`/`layer3` à la résolution de `layer1` (64×64) et on concatène sur les canaux → sortie (B, 448, 64, 64).

In [ ]:
from torchvision import models

class PaDiMFeatureExtractor(nn.Module):
    def __init__(self, backbone='resnet18'):
        super().__init__()
        net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        net.eval()
        for p in net.parameters():
            p.requires_grad = False
        self.net = net
        self._features = {}
        net.layer1.register_forward_hook(self._hook('layer1'))
        net.layer2.register_forward_hook(self._hook('layer2'))
        net.layer3.register_forward_hook(self._hook('layer3'))

    def _hook(self, name):
        def fn(_module, _inp, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x):
        self._features = {}
        _ = self.net(x)
        f1 = self._features['layer1']
        f2 = self._features['layer2']
        f3 = self._features['layer3']
        f2 = F.interpolate(f2, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        f3 = F.interpolate(f3, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        return torch.cat([f1, f2, f3], dim=1)

extractor = PaDiMFeatureExtractor(BACKBONE).to(DEVICE)

with torch.no_grad():
    sample_batch = next(iter(train_loader))['image'].to(DEVICE)
    feats = extractor(sample_batch)
print(f'Sortie features : {tuple(feats.shape)}  (B, D, H, W)')

## 3. PaDiM — fit & score

**Fit** : pour chaque position spatiale `(i, j)`, accumuler les vecteurs (après random projection sur `D_PROJ` dimensions) sur les images normales du train, puis estimer `μ_ij` et `Σ_ij + ε·I`. On stocke `μ` et `Σ⁻¹`.

**Score** : pour une image de test, distance de Mahalanobis position par position → carte (h×w) → upsample bilinéaire → lissage gaussien → heatmap (H×W).

In [ ]:
class PaDiM:
    def __init__(self, extractor, d_proj=D_PROJ, epsilon=EPSILON, device=DEVICE, seed=SEED):
        self.extractor = extractor
        self.d_proj = d_proj
        self.epsilon = epsilon
        self.device = device
        # Random projection : sélection aléatoire de d_proj canaux parmi les D du backbone
        D_total = self._infer_total_channels()
        g = torch.Generator().manual_seed(seed)
        self.proj_idx = torch.randperm(D_total, generator=g)[:d_proj].to(device)
        self.mu = None        # (HW, d')
        self.cov_inv = None   # (HW, d', d')
        self.H = self.W = self.HW = None

    def _infer_total_channels(self):
        with torch.no_grad():
            x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=self.device)
            f = self.extractor(x)
        return f.shape[1]

    @torch.no_grad()
    def _embed(self, batch):
        f = self.extractor(batch.to(self.device))      # (B, D, H, W)
        f = f.index_select(1, self.proj_idx)           # (B, d', H, W)
        B, d, H, W = f.shape
        return f.permute(0, 2, 3, 1).reshape(B, H * W, d), H, W

    def fit(self, dataloader):
        sums = None
        outers = None
        n = 0
        for batch in dataloader:
            embs, H, W = self._embed(batch['image'])
            if sums is None:
                self.H, self.W, self.HW = H, W, H * W
                sums = torch.zeros(self.HW, self.d_proj, device=self.device)
                outers = torch.zeros(self.HW, self.d_proj, self.d_proj, device=self.device)
            sums += embs.sum(dim=0)
            outers += torch.einsum('bnd,bne->nde', embs, embs)
            n += embs.shape[0]
        mu = sums / n
        cov = (outers - n * torch.einsum('nd,ne->nde', mu, mu)) / max(n - 1, 1)
        cov += self.epsilon * torch.eye(self.d_proj, device=self.device).unsqueeze(0)
        self.mu = mu
        self.cov_inv = torch.linalg.inv(cov)
        print(f'Fit OK • n={n} images • μ {tuple(self.mu.shape)} • Σ⁻¹ {tuple(self.cov_inv.shape)}')
        return self

    @torch.no_grad()
    def score(self, batch):
        embs, _, _ = self._embed(batch)
        diff = embs - self.mu.unsqueeze(0)
        tmp = torch.einsum('bnd,nde->bne', diff, self.cov_inv)
        D2 = (tmp * diff).sum(-1).clamp_min(0.0).sqrt()
        D2 = D2.reshape(-1, self.H, self.W)
        heat = F.interpolate(D2.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE),
                             mode='bilinear', align_corners=False).squeeze(1)
        return heat

## 4. Entraînement sur `cable` (train good)

In [ ]:
import time

padim = PaDiM(extractor)
t0 = time.time()
padim.fit(train_loader)
print(f'Temps de fit : {time.time() - t0:.1f}s')

## 5. Inférence sur le test set

In [ ]:
from scipy.ndimage import gaussian_filter

heatmaps, labels, gt_masks, images, paths, defect_labels = [], [], [], [], [], []

t0 = time.time()
for batch in test_loader:
    heat = padim.score(batch['image']).cpu().numpy()
    for i in range(heat.shape[0]):
        heat[i] = gaussian_filter(heat[i], sigma=SMOOTH_SIGMA)
    heatmaps.append(heat)
    labels.append(batch['label'].numpy())
    gt_masks.append(batch['mask'].squeeze(1).numpy())
    images.append(batch['image'].numpy())
    paths.extend(batch['image_path'])
    defect_labels.extend(batch['defect_label'])

heatmaps = np.concatenate(heatmaps, axis=0)
labels = np.concatenate(labels, axis=0)
gt_masks = np.concatenate(gt_masks, axis=0)
images = np.concatenate(images, axis=0)

print(f'Inférence : {len(heatmaps)} images en {time.time() - t0:.1f}s')
print(f'Heatmaps  : {heatmaps.shape}')

## 6. Évaluation — AUROC image + pixel

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Image-level : score = max de la heatmap
img_scores = heatmaps.reshape(heatmaps.shape[0], -1).max(axis=1)
auc_img = roc_auc_score(labels, img_scores)

# Pixel-level : flatten sur les images anomales (les good ont des masques nuls)
is_anom = labels == 1
y_true_pix = gt_masks[is_anom].flatten()
y_score_pix = heatmaps[is_anom].flatten()
auc_pix = roc_auc_score(y_true_pix, y_score_pix)

print(f'AUROC image-level  : {auc_img:.4f}   (référence MVTec PaDiM-RN18 cable ≈ 0.93)')
print(f'AUROC pixel-level  : {auc_pix:.4f}   (référence ≈ 0.97)')

# Courbe ROC image-level
fpr, tpr, _ = roc_curve(labels, img_scores)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color=EDA.color_anomal, lw=2, label=f'AUROC = {auc_img:.3f}')
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC image-level — cable')
ax.legend(loc='lower right')
sns.despine(ax=ax)
plt.tight_layout(); plt.show()

# AUROC par type de défaut
defect_arr = np.array(defect_labels)
rows = []
for lbl in sorted(set(defect_arr) - {'good'}):
    mask_lbl = (defect_arr == lbl) | (defect_arr == 'good')
    if (defect_arr == lbl).sum() < 1:
        continue
    auc_lbl = roc_auc_score(labels[mask_lbl], img_scores[mask_lbl])
    rows.append({'défaut': lbl, 'n': int((defect_arr == lbl).sum()), 'AUROC vs good': round(auc_lbl, 3)})
pd.DataFrame(rows).sort_values('AUROC vs good')

## 7. Visualisation des heatmaps

On affiche : (a) 2 normales (faux positifs potentiels), (b) 3 anomalies les mieux détectées, (c) 3 anomalies les moins bien détectées.

In [ ]:
anom_idx = np.where(labels == 1)[0]
norm_idx = np.where(labels == 0)[0]

norm_sorted = norm_idx[np.argsort(-img_scores[norm_idx])]   # FP les + suspects
anom_sorted = anom_idx[np.argsort(-img_scores[anom_idx])]   # mieux détectés d'abord

picks = (
    [('Normal (top-score)', i) for i in norm_sorted[:2]] +
    [('Anomal (best)',     i) for i in anom_sorted[:3]] +
    [('Anomal (worst)',    i) for i in anom_sorted[-3:]]
)

n = len(picks)
fig, axes = plt.subplots(n, 4, figsize=(13, 2.7 * n))
for r, (tag, idx) in enumerate(picks):
    img = denorm(torch.tensor(images[idx]))
    heat = heatmaps[idx]
    gt = gt_masks[idx]

    axes[r, 0].imshow(img)
    axes[r, 0].set_title(f"{tag}\n{defect_labels[idx]}  •  score={img_scores[idx]:.2f}", fontsize=9)
    axes[r, 1].imshow(heat, cmap='jet')
    axes[r, 1].set_title('Heatmap', fontsize=9)
    axes[r, 2].imshow(img)
    axes[r, 2].imshow(heat, cmap='jet', alpha=0.45)
    axes[r, 2].set_title('Overlay', fontsize=9)
    if gt.sum() > 0:
        axes[r, 3].imshow(gt, cmap='gray_r')
        axes[r, 3].set_title('GT mask', fontsize=9)
    else:
        axes[r, 3].imshow(np.zeros_like(gt), cmap='gray_r')
        axes[r, 3].set_title('(pas de mask)', fontsize=9)
    for c in range(4):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])

plt.suptitle(f'PaDiM — heatmaps sur cable  (AUROC img={auc_img:.3f}, pix={auc_pix:.3f})',
             fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## 8. Analyse → décisions de préprocessing

**À remplir après exécution :**

| Observation | Décision préprocessing |
|---|---|
| AUROC image « loin » de la référence (≈ 0.93) | revoir resize / normalisation / projection seed |
| Faux positifs sur l'arrière-plan | center crop ou seg foreground |
| Défauts manqués petits | ↑ résolution (512) ou multi-échelle |
| Heatmap diffuse | tester `wide_resnet50_2` (plus de capacité) |
| Sensibilité à l'orientation | confirmer : pas de rotations en augmentation |
| Défauts d'un type spécifique mal détectés | analyser AUROC par `label` ci-dessus |

**Prochaines étapes :**
1. Noter les observations issues des heatmaps ci-dessus.
2. Dériver la liste des transforms utiles pour `src/data/dataset.py`.
3. Re-entraîner PaDiM avec ces transforms et comparer.